In [ ]:
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from pathlib import Path
import sys

repo_root = Path.cwd().parent
sys.path.append(str(repo_root))

from myml import model_predictor_base, registry, model_predictors

/home/ioannis/dev_venv/dlenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CONFIG_PATH = repo_root / "src" / "models" / "configs.yaml"
model_registry = registry.build_model_registry(CONFIG_PATH)

In [3]:
# load model
model_name = model_registry.get_model_spec("clip").name
clip = CLIPModel.from_pretrained(model_name)
processor = CLIPProcessor.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
clip.to(device)
clip.eval()
print("Done")

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 32506.39it/s]


Done


In [4]:
# build pipeline
pipeline = (
    model_predictor_base.PipelineBuilder()
    .with_model(model_predictors.ModelCLIP(model=clip))
    .with_processor(model_predictors.ProcessorCLIP(processor=processor, device=device))
    .build()
)

In [5]:
# prepare raw data
img = Image.open(repo_root / "data" / "dog.jpeg").convert("RGB")
texts = ["a realistic photo of a dog face"]
raw_data = model_predictors.CLIPRawData(images=[img], texts=texts)

In [6]:
# run inference
output = pipeline.predict(raw_data)

# postprocessing — normalize for cosine similarity
image_embedding = output.image_output / output.image_output.norm(dim=-1, keepdim=True)
text_embedding = output.text_output / output.text_output.norm(dim=-1, keepdim=True)

In [7]:
# cosine similarity
similarity = (image_embedding @ text_embedding.T) * 100
print("\nSimilarity scores:")
for i, text in enumerate(texts):
    print(f"  '{text}': {similarity[0, i].item():.2f}")


Similarity scores:
  'a realistic photo of a dog face': 25.65


In [20]:
img_embed = output.image_output
txt_embed = output.text_output

def get_vector_norm(tensor: torch.Tensor) -> torch.Tensor:
    """
    This method is equivalent to tensor.norm(p=2, dim=-1, keepdim=True) and used to make
    model `executorch` exportable. See issue https://github.com/pytorch/executorch/issues/3566
    """
    square_tensor = torch.pow(tensor, 2)
    sum_tensor = torch.sum(square_tensor, dim=-1, keepdim=True)
    normed_tensor = torch.pow(sum_tensor, 0.5)
    return normed_tensor

img_embed = img_embed/get_vector_norm(img_embed)
txt_embed = txt_embed/get_vector_norm(txt_embed)

In [21]:
# cosine similarity
similarity = (img_embed @ txt_embed.T)
print("\nSimilarity scores:")
for i, text in enumerate(texts):
    print(f"  '{text}': {similarity[0, i].item():.2f}")


Similarity scores:
  'a realistic photo of a dog face': 0.26
